In [1]:
from pathlib import Path
import mne
import numpy as np
import pandas as pd
from mne.preprocessing import ICA


In [2]:

SCALP_CHANNELS = [
    "EEG Fp1-LE", "EEG F3-LE", "EEG C3-LE", "EEG P3-LE", "EEG O1-LE",
    "EEG F7-LE",  "EEG T3-LE", "EEG T5-LE", "EEG Fz-LE", "EEG Fp2-LE",
    "EEG F4-LE",  "EEG C4-LE", "EEG P4-LE", "EEG O2-LE", "EEG F8-LE",
    "EEG T4-LE",  "EEG T6-LE", "EEG Cz-LE", "EEG Pz-LE"
]

In [3]:
DATA_DIR = Path(r"C:\College Work\Project-1\data\raw")

file_path = DATA_DIR / "MDD S1 EC.edf"

raw = mne.io.read_raw_edf(
    file_path,
    preload=True,
    verbose=False
)

print(raw)

<RawEDF | MDD S1 EC.edf, 20 x 77568 (303.0 s), ~11.9 MiB, data loaded>


In [4]:
rename_dict = {ch: ch.replace("EEG ", "").replace("-LE", "").strip() for ch in raw.ch_names}
raw.rename_channels(rename_dict)
clean_scalp_ch = [ch.replace("EEG ", "").replace("-LE", "").strip() for ch in SCALP_CHANNELS]
raw.pick(clean_scalp_ch)

<RawEDF | MDD S1 EC.edf, 19 x 77568 (303.0 s), ~11.3 MiB, data loaded>

In [5]:
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage, on_missing='ignore')

<RawEDF | MDD S1 EC.edf, 19 x 77568 (303.0 s), ~11.3 MiB, data loaded>

In [6]:
raw.filter(l_freq=1.0, h_freq=45.0, fir_design='firwin', verbose=False)
raw.notch_filter(freqs=50.0, verbose=False)

<RawEDF | MDD S1 EC.edf, 19 x 77568 (303.0 s), ~11.3 MiB, data loaded>

In [7]:
data = raw.get_data()
ch_ptps = np.ptp(data, axis=1)
median_ptp = np.median(ch_ptps)
std_ptp = np.std(ch_ptps)

In [8]:
bad_channels = [
        raw.ch_names[i] for i, ptp in enumerate(ch_ptps) 
        if ptp > (median_ptp + 5 * std_ptp) or ptp < 1e-6
    ]
raw.info['bads'] = bad_channels

In [9]:
n_components = min(15, len(raw.ch_names) - len(bad_channels))
ica = ICA(n_components=n_components, random_state=42, max_iter=800)
ica.fit(raw, picks='eeg', reject_by_annotation=True, verbose=False)

Method,fastica
Fit parameters,algorithm=parallelfun=logcoshfun_args=Nonemax_iter=800
Fit,47 iterations on raw data (77568 samples)
ICA components,15
Available PCA components,19
Channel types,eeg
ICA components marked for exclusion,—


In [10]:
eog_indices = []
for test_ch in ['Fp1', 'Fp2']:
    if test_ch in raw.ch_names and test_ch not in raw.info['bads']:
        scores = ica.score_sources(raw, target=test_ch, score_func='pearsonr', verbose=False)
        top_matches = np.where(np.abs(scores) > 0.40)[0].tolist()
        eog_indices.extend(top_matches)

In [11]:
ica.exclude = list(set(eog_indices))[:2]
raw_cleaned = raw.copy()
ica.apply(raw_cleaned, verbose=False)


<RawEDF | MDD S1 EC.edf, 19 x 77568 (303.0 s), ~11.3 MiB, data loaded>

In [12]:
if len(raw_cleaned.info['bads']) > 0:
    raw_cleaned.interpolate_bads(reset_bads=True, verbose=False)
raw_cleaned.set_eeg_reference('average', projection=False, verbose=False)


<RawEDF | MDD S1 EC.edf, 19 x 77568 (303.0 s), ~11.3 MiB, data loaded>

In [13]:
epochs = mne.make_fixed_length_epochs(
    raw_cleaned, duration=4.0, overlap=1.0, preload=True, verbose=False
)

In [14]:
epochs.drop_bad(reject=dict(eeg=250e-6), verbose=False)

<Epochs | 59 events (all good), 0 – 3.996 s (baseline off), ~8.8 MiB, data loaded,
 '1': 59>

In [15]:
spectrum = epochs.compute_psd(method='welch', fmin=1, fmax=45, verbose=False)

In [16]:
psds, freqs = spectrum.get_data(return_freqs=True)

In [17]:
bands = {
        'delta': (1, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 45)
    }

In [18]:
total_mask = (freqs >= 1) & (freqs <= 45)
total_power = np.trapezoid(psds[:, :, total_mask], freqs[total_mask], axis=2)

In [19]:
features = []
for _, (low, high) in bands.items():
    b_mask = (freqs >= low) & (freqs < high)
    bp = np.trapezoid(psds[:, :, b_mask], freqs[b_mask], axis=2)
    rel_bp = bp / (total_power + 1e-12)
    features.append(rel_bp)

In [20]:
X_psd = np.concatenate(features, axis=1)

In [21]:
ch_names = epochs.ch_names
if 'F4' in ch_names and 'F3' in ch_names:
    f4_idx, f3_idx = ch_names.index('F4'), ch_names.index('F3')
    a_mask = (freqs >= 8) & (freqs < 13)
    alpha_f4 = np.trapezoid(psds[:, f4_idx, a_mask], freqs[a_mask], axis=1)
    alpha_f3 = np.trapezoid(psds[:, f3_idx, a_mask], freqs[a_mask], axis=1)
    faa = (np.log(alpha_f4 + 1e-12) - np.log(alpha_f3 + 1e-12))[:, None]
    X_psd = np.hstack([X_psd, faa])

In [23]:
X_psd.size


5664

In [24]:
X_psd.shape

(59, 96)